In [2]:
import os

In [3]:
# Ensure that the tokenizer is present
tokenizer_path = os.path.join("/models/gemma-3-4b-pt/1", 'tokenizer.model')
assert os.path.isfile(tokenizer_path), 'Tokenizer not found!'

In [22]:
print(dir(os.path))

['__all__', '__builtins__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_get_sep', '_joinrealpath', '_path_normpath', '_varprog', '_varprogb', 'abspath', 'altsep', 'basename', 'commonpath', 'commonprefix', 'curdir', 'defpath', 'devnull', 'dirname', 'exists', 'expanduser', 'expandvars', 'extsep', 'genericpath', 'getatime', 'getctime', 'getmtime', 'getsize', 'isabs', 'isdir', 'isfile', 'islink', 'ismount', 'join', 'lexists', 'normcase', 'normpath', 'os', 'pardir', 'pathsep', 'realpath', 'relpath', 'samefile', 'sameopenfile', 'samestat', 'sep', 'split', 'splitdrive', 'splitext', 'stat', 'supports_unicode_filenames', 'sys']


In [4]:
# Ensure that the checkpoint is present
ckpt_path = os.path.join("/models/gemma-3-4b-pt/1", f'model.ckpt')
assert os.path.isfile(ckpt_path), 'PyTorch checkpoint not found!'

In [5]:
import sys
from gemma.config import GemmaConfig, get_model_config
from gemma.model import GemmaForCausalLM
from gemma.tokenizer import Tokenizer
import contextlib
import os
import torch

In [6]:
from gemma.config import GemmaConfig, get_model_config

In [7]:
model_config = get_model_config("4b")

In [8]:
model_config.tokenizer = tokenizer_path

In [10]:
# Instantiate the model and load the weights.
torch.set_default_dtype(model_config.get_dtype())
device = torch.device("cuda")
model = GemmaForCausalLM(model_config)
model.load_weights(ckpt_path)
model = model.to(device).eval()

In [20]:
# Generate with one request in chat mode

# Chat templates
USER_CHAT_TEMPLATE = "<start_of_turn>user\n{prompt}<end_of_turn><eos>\n"
MODEL_CHAT_TEMPLATE = "<start_of_turn>model\n{prompt}<end_of_turn><eos>\n"

# Sample formatted prompt
prompt = (
    USER_CHAT_TEMPLATE.format(
        prompt='What is a good place for travel in the US?'
    )
    + MODEL_CHAT_TEMPLATE.format(prompt='California.')
    + USER_CHAT_TEMPLATE.format(prompt='What can I do in California?')
    + '<start_of_turn>model\n'
)
print('Chat prompt:\n', prompt)

output_ids = model.generate(
    USER_CHAT_TEMPLATE.format(prompt=prompt),
    device=device,
    output_len=128,
)
decoded_text = model.tokenizer.decode(output_ids)
print(decoded_text)


Chat prompt:
 <start_of_turn>user
What is a good place for travel in the US?<end_of_turn><eos>
<start_of_turn>model
California.<end_of_turn><eos>
<start_of_turn>user
What can I do in California?<end_of_turn><eos>
<start_of_turn>model

<unused56><unused46><unused2><unused25><unused24><unused1><unused20> ⁇ <unused35> ⁇ <unused41><unused42>[multimodal]<unused35><unused36><unused29><unused57><unused31><unused30><unused4><mask><unused27><unused37><unused52><unused25><unused12><unused23><unused56><unused48><unused27><unused45><unused46><unused8><unused16><unused12><unused39><unused40><unused25>[multimodal]<unused55><unused39><unused45><unused16> ⁇ <unused32><unused9><unused31><unused10><unused44><unused9><unused20><unused45><unused31><unused48><unused39><unused3><unused4><unused7><unused18><unused21><unused20><unused42><unused0><unused17><unused15><unused31><unused8><unused6><unused36><unused3><unused25><unused53><unused29><unused13><unused53><unused19><unused52><unused20><unused31><unused27

In [18]:
# Generate sample
model.generate(
    'Write a poem about an llm writing a poem.',
    device=device,
    output_len=100,
)

'<unused16><unused0><unused51><unused37><unused45><unused42><unused43><unused13><unused50>'